# Large K/V Tensor Matrix Operation Benchmark

This notebook keeps the KV-cache motivation, but the main experiment is simple: create large tensors and measure how long matrix operations take.

Why this is related to KV cache:

During decoding, attention compares the current query vector with all cached key vectors. At a simplified level, this is a matrix-vector operation:

```text
scores = K @ q
```

For long context lengths and large models, K and V tensors become very large. So before running a 100B-scale model, we can still study the core issue locally: how matrix operation latency changes as tensor size increases.


## Setup

Run this first. It picks `cuda`, `mps`, or `cpu` automatically and defines small helper functions for timing.

In [1]:
import gc
import time

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# float16 is common for inference and lets larger tensors fit in memory.
dtype = torch.float16

print("device:", device)
print("dtype:", dtype)
print("torch:", torch.__version__)


def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def bytes_to_mib(n):
    return n / (1024 ** 2)


def bench(fn, repeats=10, warmup=3):
    for _ in range(warmup):
        out = fn()
    sync_device()

    t0 = time.perf_counter()
    for _ in range(repeats):
        out = fn()
    sync_device()
    t1 = time.perf_counter()

    return (t1 - t0) / repeats, out


device: mps
dtype: torch.float16
torch: 2.4.0


## KV-Cache Size Reminder

For one layer, one K tensor has this size:

```text
batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

A full KV cache stores both K and V for every layer:

```text
2 * num_layers * batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

This section only estimates memory. The later sections measure operation time.

In [2]:
def one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size=2):
    return batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size


def full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    return 2 * num_layers * one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)


def show_kv_estimate(name, batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    one_k = one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)
    full = full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size)
    print(name)
    print(f"  one K tensor: {bytes_to_mib(one_k):,.2f} MiB")
    print(f"  full KV cache: {full / (1024 ** 3):,.2f} GiB")

show_kv_estimate(
    "SmolLM2-135M-like, L=4096",
    batch_size=1,
    cached_tokens=4096,
    num_layers=30,
    num_kv_heads=3,
    head_dim=64,
)

show_kv_estimate(
    "Larger-model-like, L=32768",
    batch_size=1,
    cached_tokens=32768,
    num_layers=80,
    num_kv_heads=8,
    head_dim=128,
)


SmolLM2-135M-like, L=4096
  one K tensor: 1.50 MiB
  full KV cache: 0.09 GiB
Larger-model-like, L=32768
  one K tensor: 64.00 MiB
  full KV cache: 10.00 GiB


## Experiment 1: Matrix-Vector Multiply

This is the simplest version of the decode attention operation.

```text
matrix: [rows, dim]
vector: [dim]
output: [rows]
```

As `rows` increases, the vector is compared with more stored rows. In KV-cache terms, this is similar to increasing the number of cached tokens.

In [9]:
dim = 128
rows_list = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 10

print(f"Matrix-vector multiply: matrix [rows, {dim}] @ vector [{dim}]")
print("rows | matrix memory | avg time")
print("-" * 46)

for rows in rows_list:
    clear_memory()
    try:
        matrix = torch.randn((rows, dim), device=device, dtype=dtype)
        vector = torch.randn((dim,), device=device, dtype=dtype)

        avg_s, out = bench(lambda: matrix @ vector, repeats=repeats)
        memory_mib = bytes_to_mib(matrix.numel() * matrix.element_size())

        print(f"{rows:6d} | {memory_mib:10.2f} MiB | {avg_s * 1000:8.3f} ms")

        del matrix, vector, out
    except RuntimeError as e:
        print(f"{rows:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-vector multiply: matrix [rows, 128] @ vector [128]
rows | matrix memory | avg time
----------------------------------------------
  1024 |       0.25 MiB |    0.147 ms
  2048 |       0.50 MiB |    0.181 ms
  4096 |       1.00 MiB |    0.189 ms
  8192 |       2.00 MiB |    0.238 ms
 16384 |       4.00 MiB |    0.256 ms
 32768 |       8.00 MiB |    0.386 ms
 65536 |      16.00 MiB |    1.395 ms
131072 |      32.00 MiB |    1.635 ms
262144 |      64.00 MiB |    1.527 ms
524288 |     128.00 MiB |    2.982 ms
1048576 |     256.00 MiB |    5.222 ms


## Experiment 2: Matrix-Matrix Multiply

This is heavier than matrix-vector multiply. It measures a more general dense operation:

```text
A: [n, dim]
B: [dim, n]
C: [n, n]
```

Be careful when increasing `n`, because the output matrix also becomes large.

In [10]:
dim = 256
n_list = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 5

print(f"Matrix-matrix multiply: A [n, {dim}] @ B [{dim}, n]")
print("n | input memory | output memory | avg time")
print("-" * 62)

for n in n_list:
    clear_memory()
    try:
        a = torch.randn((n, dim), device=device, dtype=dtype)
        b = torch.randn((dim, n), device=device, dtype=dtype)

        avg_s, c = bench(lambda: a @ b, repeats=repeats)

        input_mib = bytes_to_mib((a.numel() + b.numel()) * a.element_size())
        output_mib = bytes_to_mib(c.numel() * c.element_size())

        print(f"{n:4d} | {input_mib:10.2f} MiB | {output_mib:11.2f} MiB | {avg_s * 1000:8.3f} ms")

        del a, b, c
    except RuntimeError as e:
        print(f"{n:4d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-matrix multiply: A [n, 256] @ B [256, n]
n | input memory | output memory | avg time
--------------------------------------------------------------
 256 |       0.25 MiB |        0.12 MiB |    0.335 ms
 512 |       0.50 MiB |        0.50 MiB |    0.387 ms
1024 |       1.00 MiB |        2.00 MiB |    1.034 ms
2048 |       2.00 MiB |        8.00 MiB |    1.490 ms
4096 |       4.00 MiB |       32.00 MiB |    3.976 ms
8192 |       8.00 MiB |      128.00 MiB |   15.510 ms
16384 |      16.00 MiB |      512.00 MiB |   54.678 ms
32768 |      32.00 MiB |     2048.00 MiB | 25106.627 ms
65536 | FAILED: Invalid buffer size: 8.00 GB


## Experiment 3: K-Shaped Tensor Dot Product

This keeps the tensor closer to attention/KV-cache shape.

```text
K: [batch, kv_heads, cached_tokens, head_dim]
q: [batch, kv_heads, head_dim]
scores: [batch, kv_heads, cached_tokens]
```

This is still not a full transformer. It only times the K-side dot product that becomes expensive when the cache is large.

In [11]:
B = 1
H_kv = 8
D = 128
lengths = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 10

print("K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]")
print("L | K memory | avg time")
print("-" * 42)

for L in lengths:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        avg_s, scores = bench(lambda: torch.einsum("bhld,bhd->bhl", K, q), repeats=repeats)
        memory_mib = bytes_to_mib(K.numel() * K.element_size())

        print(f"{L:6d} | {memory_mib:8.2f} MiB | {avg_s * 1000:8.3f} ms")

        del K, q, scores
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]
L | K memory | avg time
------------------------------------------
  1024 |     2.00 MiB |    0.316 ms
  2048 |     4.00 MiB |    0.681 ms
  4096 |     8.00 MiB |    0.280 ms
  8192 |    16.00 MiB |    0.353 ms
 16384 |    32.00 MiB |    0.601 ms
 32768 |    64.00 MiB |    1.397 ms
 65536 |   128.00 MiB |    2.608 ms
131072 |   256.00 MiB |    3.799 ms
262144 |   512.00 MiB |    6.987 ms
524288 |  1024.00 MiB |   15.981 ms
1048576 |  2048.00 MiB |   42.726 ms


## Experiment 4: Layout Effect For K-Shaped Tensor

This compares the same K data in two layouts:

```text
BHLD: [batch, kv_heads, cached_tokens, head_dim]
BLHD: [batch, cached_tokens, kv_heads, head_dim]
```

Both store the same number of values. The timing can differ because memory strides and backend kernels access the data differently.

In [12]:
B = 1
H_kv = 8
D = 128
lengths = [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 10

print("Layout comparison for K-shaped dot product")
print("L | BHLD time | BLHD view time | BLHD contiguous time")
print("-" * 68)

for L in lengths:
    clear_memory()
    try:
        K_bhld = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        K_blhd_view = K_bhld.transpose(1, 2)
        K_blhd = K_blhd_view.contiguous()
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        bhld_t, _ = bench(lambda: torch.einsum("bhld,bhd->bhl", K_bhld, q), repeats=repeats)
        blhd_view_t, _ = bench(lambda: torch.einsum("blhd,bhd->bhl", K_blhd_view, q), repeats=repeats)
        blhd_t, _ = bench(lambda: torch.einsum("blhd,bhd->bhl", K_blhd, q), repeats=repeats)

        print(f"{L:6d} | {bhld_t*1000:9.3f} ms | {blhd_view_t*1000:14.3f} ms | {blhd_t*1000:18.3f} ms")

        del K_bhld, K_blhd_view, K_blhd, q
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Layout comparison for K-shaped dot product
L | BHLD time | BLHD view time | BLHD contiguous time
--------------------------------------------------------------------
  4096 |     0.242 ms |          0.248 ms |              0.925 ms
  8192 |     0.431 ms |          0.426 ms |              1.321 ms
 16384 |     0.700 ms |          0.855 ms |              3.659 ms
 32768 |     1.053 ms |          0.998 ms |              5.345 ms
 65536 |     1.659 ms |          1.859 ms |              9.673 ms
131072 |     3.721 ms |          3.690 ms |             17.656 ms
262144 |     6.448 ms |          7.106 ms |             36.597 ms
524288 |    14.537 ms |         13.191 ms |             73.172 ms
1048576 |   185.399 ms |         28.183 ms |          16524.881 ms


## What To Report

After running the notebook, report the observed trend simply:

```text
I tested large matrix operations locally without loading a large model. I measured matrix-vector multiply, matrix-matrix multiply, and a K-shaped attention-style dot product while increasing tensor size. The time generally increases as the matrix/K tensor grows, and the machine eventually becomes memory-limited. This gives a local approximation of why long-context KV caches become expensive before moving to larger models on Colab.
```

Fill in your largest successful tensor size and the timing numbers from the tables.